# Laboratório — Esperança, variância e covariância

Este laboratório calcula medidas exatas, verifica aproximações de Monte Carlo e mostra a geometria da matriz de covariância.

**Dependências:** Python 3.10+, NumPy 1.24+ e Matplotlib 3.7+.  
**Reprodutibilidade:** seed `20260907`; tolerâncias declaradas antes das simulações.


## 1. Preparação

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260907
TOL_MEDIA = 0.02
TOL_VARIANCIA = 0.06
rng = np.random.default_rng(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")


## 2. Momentos exatos de uma PMF

Para $X\in\{-2,1,4\}$, calculamos $E[X]$, $E[X^2]$, variância e desvio-padrão diretamente da PMF.


In [ ]:
x = np.array([-2.0, 1.0, 4.0])
p = np.array([0.2, 0.5, 0.3])

assert np.all(p >= 0) and np.isclose(p.sum(), 1)
media = np.sum(x * p)
segundo_momento = np.sum(x**2 * p)
variancia = np.sum((x - media)**2 * p)
variancia_atalho = segundo_momento - media**2
desvio = np.sqrt(variancia)

assert np.isclose(media, 1.3)
assert np.isclose(segundo_momento, 6.1)
assert np.isclose(variancia, 4.41)
assert np.isclose(variancia, variancia_atalho)
assert np.isclose(desvio, 2.1)
print(f"E[X]={media:.2f}; E[X²]={segundo_momento:.2f}; Var(X)={variancia:.2f}; SD(X)={desvio:.2f}")


## 3. Verificação por Monte Carlo

A simulação é uma checagem aproximada, não uma demonstração da fórmula. O tamanho e as tolerâncias são fixados antes da execução.


In [ ]:
n = 300_000
amostra = rng.choice(x, size=n, p=p)
media_mc = amostra.mean()
variancia_mc = amostra.var(ddof=0)

assert abs(media_mc - media) < TOL_MEDIA
assert abs(variancia_mc - variancia) < TOL_VARIANCIA
print(f"Média exata / simulada:     {media:.6f} / {media_mc:.6f}")
print(f"Variância exata / simulada: {variancia:.6f} / {variancia_mc:.6f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
freq = np.array([(amostra == valor).mean() for valor in x])
ax1.bar(x - 0.15, p, 0.3, label="PMF exata", color="#2563eb")
ax1.bar(x + 0.15, freq, 0.3, label="Frequência", color="#f59e0b")
ax1.axvline(media, color="#be123c", linestyle="--", label=f"E[X]={media:.1f}")
ax1.set(xlabel="x", ylabel="Massa / frequência", title="PMF e centro ponderado")
ax1.legend()
ax1.grid(axis="y", alpha=0.25)

contrib = (x - media)**2 * p
ax2.bar(x, contrib, color="#7c3aed")
ax2.set(xlabel="x", ylabel="Contribuição para a variância",
        title="(x − μ)² p(x): extremos pesam mais")
ax2.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 4. Transformação linear

Para $Y=3X-2$, a esperança é transformada linearmente e a variância recebe o quadrado do fator.


In [ ]:
y = 3 * x - 2
media_y_direta = np.sum(y * p)
var_y_direta = np.sum((y - media_y_direta)**2 * p)

assert np.isclose(media_y_direta, 3 * media - 2)
assert np.isclose(var_y_direta, 3**2 * variancia)
print(f"E[Y] = {media_y_direta:.2f}; esperado = {3*media-2:.2f}")
print(f"Var(Y) = {var_y_direta:.2f}; esperado = {9*variancia:.2f}")


## 5. Mesma esperança, dispersões diferentes

In [ ]:
a_valores, a_p = np.array([10.0]), np.array([1.0])
b_valores, b_p = np.array([0.0, 20.0]), np.array([0.5, 0.5])

def momentos_discretos(valores, massas):
    mu = np.sum(valores * massas)
    var = np.sum((valores - mu)**2 * massas)
    return mu, var

mu_a, var_a = momentos_discretos(a_valores, a_p)
mu_b, var_b = momentos_discretos(b_valores, b_p)
assert np.isclose(mu_a, mu_b) and np.isclose(var_a, 0) and np.isclose(var_b, 100)
print(f"Sistema A: E={mu_a:.1f}, Var={var_a:.1f}, SD={np.sqrt(var_a):.1f}")
print(f"Sistema B: E={mu_b:.1f}, Var={var_b:.1f}, SD={np.sqrt(var_b):.1f}")


## 6. Covariância e matriz de covariância

Geramos duas variáveis relacionadas. O ruído uniforme evita pressupor as distribuições específicas das aulas seguintes.


In [ ]:
n_xy = 8_000
x_obs = rng.uniform(-2, 2, n_xy)
ruido = rng.uniform(-0.8, 0.8, n_xy)
y_obs = 1.5 * x_obs + ruido
dados = np.column_stack([x_obs, y_obs])

sigma = np.cov(dados, rowvar=False, ddof=1)
assert sigma.shape == (2, 2)
assert np.allclose(sigma, sigma.T)
assert np.allclose(np.diag(sigma), np.var(dados, axis=0, ddof=1))
autovalores = np.linalg.eigvalsh(sigma)
assert np.all(autovalores >= -1e-12)

print("Matriz de covariância amostral:")
print(np.round(sigma, 4))
print("Autovalores:", np.round(autovalores, 6))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_obs[::8], y_obs[::8], s=12, alpha=0.3, color="#0f766e", label="amostra")
mu_xy = dados.mean(axis=0)
valores, vetores = np.linalg.eigh(sigma)
cores = ["#f59e0b", "#be123c"]
for valor, vetor, cor in zip(valores, vetores.T, cores):
    escala = 2 * np.sqrt(valor)
    ponta = mu_xy + escala * vetor
    ax.annotate("", xy=ponta, xytext=mu_xy,
                arrowprops=dict(arrowstyle="->", lw=2.5, color=cor))
ax.set(xlabel="X", ylabel="Y", title="Covariância positiva e direções da dispersão")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 7. Variância da soma

Comparamos a variância observada de $X+Y$ à identidade que inclui duas vezes a covariância.


In [ ]:
var_soma_direta = np.var(x_obs + y_obs, ddof=1)
var_soma_formula = sigma[0, 0] + sigma[1, 1] + 2 * sigma[0, 1]
assert np.isclose(var_soma_direta, var_soma_formula)
print(f"Var(X+Y) direta = {var_soma_direta:.6f}")
print(f"Pela fórmula      = {var_soma_formula:.6f}")


## 8. Covariância zero não implica independência

Defina $Y=X^2$. A covariância é zero por simetria, embora $Y$ seja determinado por $X$.


In [ ]:
x_dep = np.array([-1.0, 0.0, 1.0])
p_dep = np.array([0.25, 0.50, 0.25])
y_dep = x_dep**2
ex = np.sum(x_dep * p_dep)
ey = np.sum(y_dep * p_dep)
exy = np.sum(x_dep * y_dep * p_dep)
cov_xy = exy - ex * ey

assert np.isclose(cov_xy, 0)
assert np.array_equal(y_dep, x_dep**2)
print(f"Cov(X, X²) = {cov_xy:.1f}, mas Y é uma função determinística de X.")


## 9. Denominadores $n$ e $n-1$

In [ ]:
pequena = np.array([2.0, 4.0, 6.0])
var_n = np.var(pequena, ddof=0)
var_n1 = np.var(pequena, ddof=1)
assert np.isclose(var_n, 8/3)
assert np.isclose(var_n1, 4)
print(f"ddof=0, divisor n:   {var_n:.6f}")
print(f"ddof=1, divisor n-1: {var_n1:.6f}")


## 10. Padronização sem vazamento

Os parâmetros são ajustados no treino e apenas aplicados ao teste. Por isso o treino fica com média 0 e desvio 1, enquanto o teste não precisa ficar.


In [ ]:
feature = rng.uniform(10, 30, 1_000)
treino, teste = feature[:800], feature[800:]
mu_treino = treino.mean()
sd_treino = treino.std(ddof=0)
treino_z = (treino - mu_treino) / sd_treino
teste_z = (teste - mu_treino) / sd_treino

assert np.isclose(treino_z.mean(), 0, atol=1e-14)
assert np.isclose(treino_z.std(ddof=0), 1)
print(f"Treino padronizado: média={treino_z.mean():.6f}, SD={treino_z.std():.6f}")
print(f"Teste transformado: média={teste_z.mean():.6f}, SD={teste_z.std():.6f}")
print("Nenhum parâmetro foi reajustado com os dados de teste.")


## Conclusão

As verificações distinguiram centro, dispersão e variação conjunta; mostraram a função de `ddof`; e evitaram vazamento na padronização. Na Aula 07, aplicaremos essas ferramentas às distribuições Bernoulli, binomial, categorical e multinomial.
